# LLM Clinical Simulation Demo Pipeline

This notebook demonstrates an end-to-end workflow for:

1. Running a multi-agent clinical conversation simulation
2. Saving the generated conversation to disk
3. Evaluating the conversation with the AI judge
4. Inspecting the output in a clean, reproducible way

> Before running this notebook, make sure your `.env` file is set up with the required API keys and that the `src/` modules are present in the repository.


## 1. Imports


In [ ]:
from pathlib import Path
import json
from pprint import pprint

from dotenv import load_dotenv

from src.simulation import SimulationRunner
from src.evaluation import evaluate_conversation_file
from src.judge import JudgeConfig, load_openai_client


## 2. Load environment variables

This expects a `.env` file in the project root containing your API keys, for example:

```text
OPENAI_API_KEY=your_key_here
ANTHROPIC_API_KEY=your_key_here
GOOGLE_API_KEY=your_key_here
```


In [ ]:
load_dotenv(override=True)
print('Environment loaded.')


## 3. Run one simulation


In [ ]:
runner = SimulationRunner()
conversation_path = runner.run_and_save(verbose=True)
print(f'Conversation saved to: {conversation_path}')


## 4. Inspect the saved conversation JSON


In [ ]:
conversation_path = Path(conversation_path)
with conversation_path.open('r', encoding='utf-8') as f:
    conversation_payload = json.load(f)

print('Top-level keys:', list(conversation_payload.keys()))
print('Conversation ID:', conversation_payload.get('conversation_id'))
print('Number of turns:', conversation_payload.get('num_turns'))


## 5. Preview the conversation transcript


In [ ]:
for turn in conversation_payload['turns']:
    print(f"{turn['speaker'].upper()}: {turn['text']}")
    print()


## 6. Evaluate the conversation with the AI judge

This uses the OpenAI-based judge pipeline defined in `src/judge.py` and `src/evaluation.py`.


In [ ]:
judge_client = load_openai_client()
judge_config = JudgeConfig(model='gpt-4o', temperature=0.0, max_retries=3)

evaluation_record = evaluate_conversation_file(
    json_file=conversation_path,
    client=judge_client,
    config=judge_config,
    output_jsonl=Path('data/judge_results.jsonl'),
    per_file_output_dir=Path('data/judge_results'),
)

print('Evaluation complete.')


## 7. Inspect the evaluation result


In [ ]:
pprint(evaluation_record)


## 8. Optional: inspect the saved per-file judge output


In [ ]:
judge_results_dir = Path('data/judge_results')
judge_files = sorted(judge_results_dir.glob('*.judge.json'))

if judge_files:
    print('Saved judge result files:')
    for jf in judge_files:
        print('-', jf)
else:
    print('No per-file judge outputs found yet.')


## 9. Next steps

You can extend this notebook to:

- run batch simulations
- compare different model combinations
- evaluate structured vs unstructured communication prompts
- export results for statistical analysis
